# ECG Interval Feature Extraction (Tier 1 — 6 Baseline Features)

Extracts 6 interval-based features from digitized 12-lead ECG NPZ files using NeuroKit2.
All features are extracted from **lead II** (standard limb lead for rhythm analysis).

| Feature | Description | Lead | Clinical Meaning |
|---|---|---|---|
| `rr_ms` | Median R-R interval (ms) | II | Heart rate proxy (HR = 60000/RR) |
| `rr_std_ms` | Std dev of R-R intervals (ms) | II | Heart rate variability; high = irregular rhythm (AF) |
| `p_dur_ms` | Median P-wave duration (ms) | II | Atrial depolarization time; prolonged in LA enlargement |
| `qrs_dur_ms` | Median QRS duration (ms) | II | Ventricular depolarization time |
| `qtc_ms` | Corrected QT interval — Bazett (ms) | II | Depol + repol duration, normalized to HR |
| `sokolow` | S(V1) + R(V5) amplitude (µV) | V1, V5 | Sokolow–Lyon LVH criterion |

**Output:** `ecg_intervals.csv` — used by all `*_iv` and `*_ext` model variants.

**Note:** `sokolow` requires lead V1 and V5, all other features use lead II only.

In [ ]:
import os, warnings, time
os.environ['TORCHDYNAMO_DISABLE'] = '1'
warnings.filterwarnings('ignore')

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import neurokit2 as nk
from tqdm.auto import tqdm

# ── Paths (adjust for your environment) ──
PROJECT_ROOT = Path('/kaggle/input/datasets/pascalhafidzfajri/mlcv-mitral')
NPZ_DIR      = PROJECT_ROOT / 'mitral_images' / 'mitral_images'
SPLIT_CSV    = PROJECT_ROOT / 'data_split_3level_exp10.csv'
OUT_CSV      = Path('.') / 'ecg_intervals.csv'
CHECKPOINT   = Path('.') / 'intervals_checkpoint.csv'

FS          = 500   # sampling rate Hz
N_WORKERS   = 4
SAVE_EVERY  = 500

LEADS = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
LEAD_IDX = {l: i for i, l in enumerate(LEADS)}

print('neurokit2:', nk.__version__)
print('NPZ_DIR exists:', NPZ_DIR.exists())

## 1. Load sample list

In [ ]:
split = pd.read_csv(SPLIT_CSV)
split['stem'] = split['filename'].map(lambda x: Path(str(x)).stem)
split['npz_path'] = split['stem'].map(lambda s: str(NPZ_DIR / f'{s}.npz'))
split['npz_exists'] = split['npz_path'].map(lambda p: Path(p).exists())
avail = split[split['npz_exists']].copy().reset_index(drop=True)
print(f'Available: {len(avail)} / {len(split)} samples')

done_stems = set()
if CHECKPOINT.exists():
    done_df = pd.read_csv(CHECKPOINT)
    done_stems = set(done_df['stem'].astype(str))
    print(f'Checkpoint: {len(done_stems)} already done')

todo = avail[~avail['stem'].astype(str).isin(done_stems)].copy().reset_index(drop=True)
print(f'To process: {len(todo)} samples')

## 2. Extraction function

In [ ]:
def extract_intervals(npz_path: str, stem: str) -> dict:
    """
    Extract 6 interval features from one ECG sample.
    
    Pipeline:
      1. Load NPZ, apply overlap-zeroing
      2. Clean lead II with NeuroKit2
      3. Detect R-peaks → compute RR intervals
      4. Delineate P/QRS/T waves → compute durations
      5. Compute QTc via Bazett formula
      6. Compute Sokolow–Lyon index from V1 + V5
    """
    result = {
        'stem': stem,
        'rr_ms': np.nan,
        'rr_std_ms': np.nan,
        'p_dur_ms': np.nan,
        'qrs_dur_ms': np.nan,
        'qtc_ms': np.nan,
        'sokolow': np.nan,
    }
    
    try:
        with np.load(npz_path, allow_pickle=False) as d:
            sig = d['signal_matrix'].astype(np.float32)   # (T, 12)
            mask = d['overlap_mask'].astype(bool)          # (T, 12)
        sig[mask] = 0.0
        
        # ── Lead II: intervals ──
        lead_ii = sig[:, LEAD_IDX['II']].astype(np.float64)
        clean_ii = nk.ecg_clean(lead_ii, sampling_rate=FS)
        
        _, rpeaks_d = nk.ecg_peaks(clean_ii, sampling_rate=FS, method='pantompkins1985')
        rp = rpeaks_d['ECG_R_Peaks']
        
        if len(rp) < 3:
            return result
        
        # RR intervals
        rr = np.diff(rp) / FS * 1000  # ms
        result['rr_ms']     = float(np.median(rr))
        result['rr_std_ms'] = float(np.std(rr))
        
        # Wave delineation
        _, waves = nk.ecg_delineate(clean_ii, rpeaks_d, sampling_rate=FS,
                                     method='dwt', show=False)
        
        # P-wave duration
        p_on  = np.array(waves.get('ECG_P_Onsets',  []), dtype=float)
        p_off = np.array(waves.get('ECG_P_Offsets', []), dtype=float)
        if len(p_on) > 0 and len(p_off) > 0:
            n = min(len(p_on), len(p_off))
            p_durs = (p_off[:n] - p_on[:n]) / FS * 1000
            valid = p_durs[np.isfinite(p_durs) & (p_durs > 0) & (p_durs < 300)]
            if len(valid) >= 2:
                result['p_dur_ms'] = float(np.median(valid))
        
        # QRS duration
        q_on  = np.array(waves.get('ECG_R_Onsets',  []), dtype=float)
        s_off = np.array(waves.get('ECG_R_Offsets', []), dtype=float)
        if len(q_on) > 0 and len(s_off) > 0:
            n = min(len(q_on), len(s_off))
            qrs_durs = (s_off[:n] - q_on[:n]) / FS * 1000
            valid = qrs_durs[np.isfinite(qrs_durs) & (qrs_durs > 0) & (qrs_durs < 300)]
            if len(valid) >= 2:
                result['qrs_dur_ms'] = float(np.median(valid))
        
        # QTc (Bazett): QTc = QT / sqrt(RR_sec)
        t_off = np.array(waves.get('ECG_T_Offsets', []), dtype=float)
        if len(q_on) > 0 and len(t_off) > 0:
            n = min(len(q_on), len(t_off))
            qt_ms = (t_off[:n] - q_on[:n]) / FS * 1000
            valid_qt = qt_ms[np.isfinite(qt_ms) & (qt_ms > 100) & (qt_ms < 700)]
            if len(valid_qt) >= 2:
                med_qt = float(np.median(valid_qt))
                rr_sec = result['rr_ms'] / 1000 if result['rr_ms'] > 0 else np.nan
                if np.isfinite(rr_sec) and rr_sec > 0:
                    result['qtc_ms'] = med_qt / np.sqrt(rr_sec)
        
        # ── Sokolow–Lyon: |S(V1)| + R(V5) ──
        lead_v1 = sig[:, LEAD_IDX['V1']].astype(np.float64)
        lead_v5 = sig[:, LEAD_IDX['V5']].astype(np.float64)
        clean_v1 = nk.ecg_clean(lead_v1, sampling_rate=FS)
        clean_v5 = nk.ecg_clean(lead_v5, sampling_rate=FS)
        
        # S-wave amplitude in V1
        _, rp_v1 = nk.ecg_peaks(clean_v1, sampling_rate=FS, method='pantompkins1985')
        _, waves_v1 = nk.ecg_delineate(clean_v1, rp_v1, sampling_rate=FS, method='dwt', show=False)
        s_v1_locs = np.array(waves_v1.get('ECG_S_Peaks', []), dtype=float)
        s_v1_valid = s_v1_locs[np.isfinite(s_v1_locs)].astype(int)
        s_v1_valid = s_v1_valid[(s_v1_valid >= 0) & (s_v1_valid < len(clean_v1))]
        sv1_amp = abs(float(np.median(clean_v1[s_v1_valid]))) if len(s_v1_valid) >= 2 else np.nan
        
        # R-wave amplitude in V5
        _, rp_v5 = nk.ecg_peaks(clean_v5, sampling_rate=FS, method='pantompkins1985')
        rv5_locs = rp_v5['ECG_R_Peaks']
        rv5_amp = float(np.median(clean_v5[rv5_locs])) if len(rv5_locs) >= 2 else np.nan
        
        if np.isfinite(sv1_amp) and np.isfinite(rv5_amp):
            result['sokolow'] = sv1_amp + rv5_amp
    
    except Exception:
        pass
    
    return result


# Smoke test
if len(avail) > 0:
    t0 = time.time()
    r = extract_intervals(avail.iloc[0]['npz_path'], str(avail.iloc[0]['stem']))
    print(f'Smoke test ({time.time()-t0:.2f}s):')
    for k, v in r.items():
        if k != 'stem':
            print(f'  {k}: {v}')

## 3. Run extraction (parallelized)

In [ ]:
all_results = list(pd.read_csv(CHECKPOINT).to_dict('records')) if CHECKPOINT.exists() else []

batch_buffer = []
t_start = time.time()

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = {
        ex.submit(extract_intervals, str(row['npz_path']), str(row['stem'])): i
        for i, row in todo.iterrows()
    }
    for future in tqdm(as_completed(futures), total=len(futures),
                       desc='extracting ECG intervals'):
        res = future.result()
        batch_buffer.append(res)
        all_results.append(res)
        
        if len(batch_buffer) >= SAVE_EVERY:
            pd.DataFrame(all_results).to_csv(CHECKPOINT, index=False)
            batch_buffer = []

df = pd.DataFrame(all_results)
df.to_csv(CHECKPOINT, index=False)

elapsed = time.time() - t_start
print(f'Done: {len(df)} samples in {elapsed/60:.1f} min')

## 4. Quality check and save

In [ ]:
INTERVAL_COLS = ['rr_ms', 'rr_std_ms', 'p_dur_ms', 'qrs_dur_ms', 'qtc_ms', 'sokolow']

print('=== NaN rates ===')
print(df[INTERVAL_COLS].isna().mean().round(4).to_string())

print('\n=== Descriptive statistics ===')
print(df[INTERVAL_COLS].describe().round(1).to_string())

# Sanity checks
print('\n=== Sanity checks ===')
med_rr = df['rr_ms'].median()
print(f'Median HR: {60000/med_rr:.0f} bpm (expect 60-100)')  
print(f'Median QRS: {df["qrs_dur_ms"].median():.0f} ms (expect 80-120)')
print(f'Median QTc: {df["qtc_ms"].median():.0f} ms (expect 350-470)')
print(f'Median Sokolow: {df["sokolow"].median():.0f} µV')

# Save final
df.to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV} ({len(df)} rows, {len(df.columns)} cols)')

## 5. Per-class distribution

Verify that features show expected clinical patterns across Control/MR/MS.

In [ ]:
split2 = pd.read_csv(SPLIT_CSV)
split2['stem'] = split2['filename'].map(lambda x: Path(str(x)).stem)
label_map = {'Kontrol': 'Control', 'MR': 'MR', 'MS': 'MS'}
merged = df.merge(split2[['stem', 'flat_label']], on='stem', how='left')
merged['class'] = merged['flat_label'].map(label_map)

print(f"{'Feature':12s}  {'Control':>10s}  {'MR':>10s}  {'MS':>10s}  {'Note'}")
print('-' * 70)
for col in INTERVAL_COLS:
    vals = {c: merged[merged['class']==c][col].median() for c in ['Control','MR','MS']}
    print(f"{col:12s}  {vals['Control']:10.1f}  {vals['MR']:10.1f}  {vals['MS']:10.1f}")

print('\n=== Expected patterns ===')
print('rr_ms:     MS < Control (MS often has AF → faster HR)')
print('rr_std_ms: MS > Control (AF → irregular RR)')
print('p_dur_ms:  MS > Control (LA enlargement → P mitrale)')
print('sokolow:   MR > Control (volume overload → LVH)')